<h1 style="color:#1B4F72; font-weight:bold;text-align:center;">
Stage 2 Modeling for Toluene vs 2-Butanone Probability Estimation
</h1>

<h2 style="color:#2E86C1; font-weight:bold;">
1. Introduction
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This notebook presents the second stage of the hierarchical gas sensor classification framework. At this stage, the modeling problem is restricted to <b>true single-gas samples</b>, so that the learning task can focus specifically on gas-type discrimination within the single-gas branch.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The main objective of this stage is to estimate the probability that a single-gas sample belongs to the 
<b>Toluene</b> or <b>2-butanone</b> class. Instead of producing only a hard class label, the models in this notebook 
are designed to output <b>class probability estimates</b>, which provide a quantitative measure of prediction confidence.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This probability-based formulation is particularly important in sensor-based classification problems, where signal 
patterns may partially overlap. In such cases, confidence scores enable more informed decision-making compared to 
binary predictions alone.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The notebook follows a structured workflow similar to the first-stage modeling notebook, including data loading, 
target definition, group-aware validation, model comparison, and probability-based interpretation of the final model. <i>(Literature Review, Section 3.2, Feature Engineering for Gas Sensor Time-Series)</i>
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.1 Modeling Objective
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The objective of this notebook is to build and compare <b>Stage 2</b> models that estimate whether a single-gas sample belongs to the <code>Toluene</code> or <code>2-butanone</code> class. Unlike Stage 1, which acts as a routing classifier between <code>single_gas</code> and <code>mixture</code>, this stage focuses on finer discrimination within the single-gas branch.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Because the output of this stage may be used in downstream decision-making, probability estimation is treated as a 
core modeling objective rather than an optional extension. The goal is not only to classify samples correctly, but 
also to provide <b>well-calibrated confidence scores</b> that reflect the uncertainty of each prediction.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.2 CRISP-DM Context
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Within CRISP-DM, this notebook belongs mainly to the <b>Modeling</b> and <b>Evaluation</b> phases. It reuses the engineered feature representation created earlier and investigates how well different candidate models can separate the two single-gas classes under a leakage-aware validation setup.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The expected output of this phase is a justified Stage 2 model choice together with a clear understanding of how reliable the resulting probability estimates are on unseen runs.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.3 Relationship to Stage 1
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Conceptually, Stage 2 is executed only after a sample has been routed into the single-gas branch by Stage 1. However, for model development in this notebook, the training data is created from the <b>true labeled single-gas subset</b> rather than from Stage 1 predictions. This keeps the second-stage learning problem well defined and avoids injecting first-stage routing errors into the model development process.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In a full deployment scenario, Stage 2 would operate on the subset of samples predicted as single-gas by Stage 1. 
However, during model development, isolating Stage 2 using ground-truth labels allows a cleaner evaluation of its 
intrinsic classification capability.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This distinction is important for interpreting the results: the performance reported here reflects the intrinsic difficulty of discriminating between Toluene and 2-butanone once true single-gas samples are available, not the full end-to-end performance of the complete two-stage pipeline.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.4 Dataset and Split Strategy
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The input data for this notebook consists of feature-engineered windows extracted from runs that belong to the true single-gas subset. Although each row is a window-level sample, the evaluation strategy must remain <b>run-aware</b>. Windows originating from the same run are not independent and should stay grouped together during validation. <i>(Literature Review, Section 5, Evaluation Strategy for Gas Sensor Classification)</i>
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This is particularly important in Stage 2, where the number of available runs is more limited than in the original full dataset. A leakage-safe split is therefore necessary to avoid overstating how well the model generalizes to unseen single-gas experiments.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.5 Evaluation Criteria and Probability Use
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Model assessment in this notebook goes beyond hard-label accuracy. In addition to accuracy, <b>precision</b>, <b>recall</b>, and <b>F1-score</b> are used to evaluate class-specific behavior and overall balance. <i>(Literature Review, Section 5, Evaluation Strategy for Gas Sensor Classification)</i>
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, special attention is given to the <b>quality of probability estimates</b>. Reliable probability outputs are essential for interpreting model confidence and for supporting downstream decisions in the hierarchical pipeline.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.6 Model Selection Principle
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The final Stage 2 model should be selected based on both predictive performance and the quality of its probability-based interpretation. A suitable model is not simply the one with the highest single score, but the one that offers the best balance between discrimination, stability across folds, and useful confidence estimates for downstream use.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The selected model should therefore not only achieve strong classification performance, but also produce stable and 
interpretable probability estimates. This dual requirement ensures that Stage 2 contributes both accurate predictions 
and meaningful confidence information to the overall system.
</p>

<h2 style="color:#2E86C1; font-weight:bold;">
2. Data Loading
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The modeling process begins by loading the <b>single-gas subset</b> prepared in the previous notebook. This dataset 
contains only true single-gas samples and serves as the input for Stage 2 probability estimation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Each row corresponds to a fixed-length window extracted from the original sensor signal and is described using the 
same engineered statistical, temporal, and shape-based features developed in Stage 1. This ensures consistency 
between stages and allows direct reuse of the established feature representation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Because this dataset is derived from the ground-truth single-gas subset, it provides a clean and well-defined input 
for modeling, enabling reliable evaluation of gas-type discrimination without interference from Stage 1 classification errors.
</p>

In [ ]:
import pandas as pd

single_df = pd.read_parquet("../data/processed/features_single_gas.parquet")

print("Dataset shape:", single_df.shape)
display(single_df.head())

Dataset shape: (1194, 44)


,run_id,experiment,experiment_folder,run_folder,repeat_index,window_id,start_idx,end_idx,time_start,time_end,...,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position,target_mixture
0,UV ink - 2-butanone__run_1,UV ink - 2-butanone,UV ink - 2-butanone,..\data\raw\CMOS_addtional_datasets\UV ink - 2...,1.0,UV ink - 2-butanone__run_1_w0,0,99,5.232,124.808,...,0.053367,-0.615725,0.040404,0.102041,0.122550,0.079291,0.525253,0.474747,0.525253,0
1,UV ink - 2-butanone__run_1,UV ink - 2-butanone,UV ink - 2-butanone,..\data\raw\CMOS_addtional_datasets\UV ink - 2...,1.0,UV ink - 2-butanone__run_1_w50,50,149,66.727,188.132,...,0.802705,1.052089,0.191919,0.132653,0.198287,0.122610,0.020202,0.979798,0.020202,0
2,UV ink - 2-butanone__run_1,UV ink - 2-butanone,UV ink - 2-butanone,..\data\raw\CMOS_addtional_datasets\UV ink - 2...,1.0,UV ink - 2-butanone__run_1_w100,100,199,125.951,253.576,...,0.479149,-0.539676,0.202020,0.122449,0.084509,0.159318,0.717172,0.282828,0.717172,0
3,UV ink - 2-butanone__run_1,UV ink - 2-butanone,UV ink - 2-butanone,..\data\raw\CMOS_addtional_datasets\UV ink - 2...,1.0,UV ink - 2-butanone__run_1_w150,150,249,189.476,319.656,...,0.689765,-0.149542,0.040404,0.091837,0.121286,0.168425,0.777778,0.222222,0.777778,0
4,UV ink - 2-butanone__run_1,UV ink - 2-butanone,UV ink - 2-butanone,..\data\raw\CMOS_addtional_datasets\UV ink - 2...,1.0,UV ink - 2-butanone__run_1_w200,200,299,254.932,384.432,...,0.497454,-0.067748,0.090909,0.112245,0.164799,0.191332,0.272727,0.727273,0.272727,0


<h2 style="color:#2E86C1; font-weight:bold;">
3. Dataset Preparation
</h2>

<h3 style="color:#2874A6; font-weight:bold;">
3.1 Defining Features, Target, and Group Information
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To prepare the dataset for Stage 2 modeling, a binary target variable is created to represent the gas type of each 
single-gas sample. The classification task focuses on distinguishing between <b>Toluene</b> and <b>2-butanone</b>, 
which are encoded as separate classes based on the experiment labels.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
At the same time, metadata columns are excluded from the feature matrix, while <b>run_id</b> is preserved separately 
as group information for leakage-aware evaluation.
</p>

In [3]:
def map_gas_type(exp_name):
    exp_name = str(exp_name).lower()

    if "toluene" in exp_name:
        return 1
    elif "2-butanone" in exp_name or "butanone" in exp_name:
        return 0
    else:
        return None

single_df["target_gas_type"] = single_df["experiment"].apply(map_gas_type)

# remove undefined labels if any
single_df = single_df.dropna(subset=["target_gas_type"]).copy()
single_df["target_gas_type"] = single_df["target_gas_type"].astype(int)

# define target and groups
y = single_df["target_gas_type"]
groups = single_df["run_id"]

# define features
X = single_df.drop(columns=[
    "experiment",
    "target_gas_type",
    "run_id",
    "experiment_folder",
    "run_folder",
    "repeat_index",
    "window_id"
])

X = X.drop(columns=["start_idx", "end_idx", "time_start", "time_end"])

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of groups:", groups.nunique())

print("\nClass distribution:")
print(y.value_counts())

display(X.head())

Feature matrix shape: (1194, 34)
Target shape: (1194,)
Number of groups: 8

Class distribution:
target_gas_type
1    621
0    573
Name: count, dtype: int64


,mean,std,min,max,median,range,iqr,cv,first_diff_mean,first_diff_std,...,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position,target_mixture
0,0.118204,0.043978,0.032538,0.213033,0.120156,0.180495,0.066877,0.372049,-0.000437,0.009917,...,0.053367,-0.615725,0.040404,0.102041,0.122550,0.079291,0.525253,0.474747,0.525253,0
1,0.114223,0.034240,0.051580,0.213033,0.112207,0.161453,0.039510,0.299760,-0.000764,0.006635,...,0.802705,1.052089,0.191919,0.132653,0.198287,0.122610,0.020202,0.979798,0.020202,0
2,0.137692,0.033306,0.084509,0.213133,0.133337,0.128624,0.049113,0.241887,0.000756,0.004997,...,0.479149,-0.539676,0.202020,0.122449,0.084509,0.159318,0.717172,0.282828,0.717172,0
3,0.168061,0.052866,0.076713,0.288687,0.163603,0.211974,0.067428,0.314561,0.000476,0.009811,...,0.689765,-0.149542,0.040404,0.091837,0.121286,0.168425,0.777778,0.222222,0.777778,0
4,0.175722,0.050107,0.076713,0.288687,0.170516,0.211974,0.057747,0.285148,0.000268,0.009734,...,0.497454,-0.067748,0.090909,0.112245,0.164799,0.191332,0.272727,0.727273,0.272727,0



<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The resulting feature matrix contains <b>1194 samples</b> and <b>34 numerical features</b>, where each sample corresponds 
to a fixed-length window extracted from single-gas sensor signals. All retained features are derived directly from the 
signal, as metadata and temporal indexing variables have been excluded to ensure a purely signal-driven representation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The target variable represents gas type, with <b>1 corresponding to Toluene</b> and <b>0 corresponding to 2-butanone</b>. 
The class distribution is relatively balanced, with 621 samples for Toluene and 573 samples for 2-butanone. This balance 
supports stable model training and reduces the risk of bias toward a dominant class.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The dataset includes <b>8 unique runs</b>, preserved through the <b>run_id</b> grouping variable. This grouping structure is 
essential because multiple windows originate from the same experimental run and are therefore not independent observations. 
Maintaining this information enables leakage-aware evaluation strategies in subsequent modeling steps.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the dataset is well-prepared for Stage 2 modeling. It provides a clean and fully numerical feature space, a clearly 
defined binary target variable, and appropriate group identifiers, ensuring that the modeling process can focus on learning 
meaningful signal patterns while preserving a robust and realistic evaluation framework.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
3.2 Group-Aware Cross-Validation Strategy
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
A leakage-aware validation strategy is essential for the current dataset because multiple windowed samples are extracted 
from the same experimental run. Since these samples are not independent, using a conventional random split would lead 
to overly optimistic and misleading performance estimates.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To address this issue, <b>StratifiedGroupKFold</b> is used as the primary cross-validation strategy. This method ensures 
that the class distribution is preserved across folds while simultaneously enforcing that all samples from the same 
<b>run_id</b> remain within a single fold. As a result, the model is always evaluated on completely unseen runs, 
maintaining a realistic and leakage-safe evaluation setting.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Given the relatively small number of available runs in the single-gas subset (<b>8 runs</b>), a <b>3-fold</b> cross-validation 
setup is selected. This configuration provides a practical balance between reliable validation and efficient use of the 
available data, ensuring that each fold contains a sufficient number of runs for both training and validation.
</p>

In [4]:
from sklearn.model_selection import StratifiedGroupKFold

# Define group-aware cross-validation strategy
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print("Number of folds:", cv.get_n_splits(X, y, groups))

Number of folds: 3


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The cross-validation setup confirms that the dataset is partitioned into <b>3 folds</b>, consistent with the intended 
evaluation design. In each fold, samples originating from the same run are grouped together, ensuring that no information 
from a given run appears in both training and validation subsets simultaneously.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This grouping constraint is particularly important in the current context, where signal patterns within a single run may 
be highly correlated. By enforcing run-level separation, the evaluation more accurately reflects the model’s ability to 
generalize to entirely unseen experimental conditions.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the selected cross-validation strategy provides a robust and realistic framework for model assessment, reducing 
the risk of data leakage and ensuring that reported performance metrics are representative of real-world deployment scenarios.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
3.3 Evaluation Metrics
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To evaluate the performance of the machine learning models, multiple classification metrics are used to capture both 
overall predictive performance and class-specific behavior. Since the current task is a <b>binary classification problem</b> 
focused on distinguishing <b>Toluene</b> and <b>2-butanone</b>, <b>classification accuracy</b> is used as a general 
indicator of overall model performance.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
However, accuracy alone may not fully reflect model quality, especially when evaluating the reliability of predictions 
for each gas type. Therefore, additional metrics including <b>precision</b>, <b>recall</b>, and <b>F1-score</b> are also 
considered. These metrics provide a more detailed understanding of how well the model distinguishes between the two gases, 
particularly in terms of balancing false positives and false negatives.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Given that Stage 2 focuses on probability estimation and reliable discrimination between similar signal patterns, 
the <b>F1-score</b> is treated as a key metric for model comparison. It provides a balanced measure of precision and recall, 
making it especially suitable for evaluating classification performance when both types of errors are important.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Under the group-aware cross-validation framework, model performance is evaluated across multiple folds. The final results 
are reported as the <b>mean score</b> together with the <b>standard deviation</b> across folds, providing a robust and 
stable estimate of model generalization on unseen runs.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
4.1 Logistic Regression (Baseline Model)
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Logistic Regression is used as the baseline model for stage-2 gas-type classification. The objective is to evaluate 
whether the engineered features contain sufficient discriminative information to distinguish between <b>Toluene</b> 
and <b>2-butanone</b> samples within the single-gas subset. <i>(Literature Review, Section 4, Machine Learning Models for Gas Sensor Classification)</i>
</p>

In [5]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

# Logistic Regression pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Cross-validation with multiple metrics
lr_results = cross_validate(
    lr_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False
)

# Summary table
lr_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        lr_results["test_accuracy"].mean(),
        lr_results["test_precision"].mean(),
        lr_results["test_recall"].mean(),
        lr_results["test_f1"].mean()
    ],
    "Std": [
        lr_results["test_accuracy"].std(),
        lr_results["test_precision"].std(),
        lr_results["test_recall"].std(),
        lr_results["test_f1"].std()
    ]
})

display(lr_summary)

,Metric,Mean,Std
0,Accuracy,0.742159,0.180948
1,Precision,0.569272,0.416967
2,Recall,0.595628,0.430065
3,F1-score,0.581953,0.423116


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The Logistic Regression model achieves a <b>mean accuracy of 74.2%</b> and an <b>F1-score of 58.2%</b>, indicating that 
the classification task in Stage 2 is significantly more challenging than the Stage 1 mixture detection problem.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
A key observation is the relatively low F1-score compared to accuracy, which suggests an imbalance between precision 
and recall. This indicates that the model struggles to consistently distinguish between <b>Toluene</b> and 
<b>2-butanone</b>, likely due to overlapping signal patterns between the two gases.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, the relatively high <b>standard deviation</b> across metrics reflects variability between folds. This is 
expected given the limited number of runs (<b>8 runs</b>) and highlights the importance of group-aware validation 
for obtaining realistic performance estimates.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the baseline results suggest that while the engineered features provide some discriminative power, 
linear decision boundaries are not sufficient to fully capture the complexity of the problem. This motivates 
the evaluation of more flexible, non-linear models in the following sections.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
4.2 Support Vector Machine (SVM)
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Support Vector Machine (SVM) is evaluated as a more flexible alternative to the linear baseline model. By using the 
<b>Radial Basis Function (RBF)</b> kernel, SVM can capture non-linear relationships and more complex decision boundaries 
within the engineered feature space.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This capability is particularly relevant for the current task, where distinguishing between <b>Toluene</b> and 
<b>2-butanone</b> may depend on subtle non-linear differences in temporal, statistical, and shape-based signal descriptors 
that cannot be fully captured by linear models.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Because SVM is sensitive to feature scaling, the model is implemented within a pipeline that includes 
<b>StandardScaler</b>, ensuring that scaling is applied within each cross-validation fold and preventing data leakage. 
In addition, probability estimation is enabled so that the trained model can provide confidence scores for gas-type prediction.
</p>

In [6]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

# SVM pipeline
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", probability=True, random_state=42))
])

# Cross-validation with multiple metrics
svm_results = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False
)

# Summary table
svm_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        svm_results["test_accuracy"].mean(),
        svm_results["test_precision"].mean(),
        svm_results["test_recall"].mean(),
        svm_results["test_f1"].mean()
    ],
    "Std": [
        svm_results["test_accuracy"].std(),
        svm_results["test_precision"].std(),
        svm_results["test_recall"].std(),
        svm_results["test_f1"].std()
    ]
})

display(svm_summary)

,Metric,Mean,Std
0,Accuracy,0.733890,0.187002
1,Precision,0.577860,0.421504
2,Recall,0.557377,0.416234
3,F1-score,0.567070,0.418356


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The SVM model achieves a <b>mean accuracy of 73.4%</b> and an <b>F1-score of 56.7%</b>, which is slightly lower than 
the Logistic Regression baseline. This indicates that introducing a non-linear decision boundary does not lead to an 
improvement in classification performance for the current feature space.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Similar to the baseline model, there is a noticeable gap between accuracy and F1-score, suggesting that the model 
struggles to balance precision and recall effectively. This reflects the inherent difficulty of distinguishing between 
<b>Toluene</b> and <b>2-butanone</b>, where signal characteristics may overlap significantly.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, the relatively high <b>standard deviation</b> across evaluation metrics indicates considerable variability 
between folds. This suggests that model performance is sensitive to the specific runs included in training and validation, 
which is expected given the limited number of runs in the dataset.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results suggest that increasing model complexity alone does not improve performance in this stage. The 
engineered features may not provide sufficiently distinct patterns for reliable separation using SVM, motivating further 
exploration of alternative models or feature representations.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
4.3 Random Forest
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Random Forest is evaluated as an ensemble-based model that combines multiple decision trees to improve predictive 
performance and robustness. Unlike linear models, it can naturally capture non-linear relationships and complex 
feature interactions without requiring explicit feature transformations.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This capability is particularly relevant for the current task, where distinguishing between <b>Toluene</b> and 
<b>2-butanone</b> depends on subtle variations in temporal, statistical, and shape-based signal descriptors. By 
aggregating multiple decision trees, Random Forest can model heterogeneous patterns within the feature space more effectively.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, Random Forest is less sensitive to feature scaling and generally more robust to noise and variability across 
runs. This makes it a strong candidate for handling the inherent variability present in sensor-based datasets.
</p>

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
import pandas as pd

# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Cross-validation with multiple metrics
rf_results = cross_validate(
    rf_model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False,
    n_jobs=-1
)

# Summary table
rf_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        rf_results["test_accuracy"].mean(),
        rf_results["test_precision"].mean(),
        rf_results["test_recall"].mean(),
        rf_results["test_f1"].mean()
    ],
    "Std": [
        rf_results["test_accuracy"].std(),
        rf_results["test_precision"].std(),
        rf_results["test_recall"].std(),
        rf_results["test_f1"].std()
    ]
})

display(rf_summary)

,Metric,Mean,Std
0,Accuracy,0.762813,0.174510
1,Precision,0.590547,0.426736
2,Recall,0.597814,0.431047
3,F1-score,0.594152,0.428871


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The Random Forest model achieves a <b>mean accuracy of 76.3%</b> and an <b>F1-score of 59.4%</b>, representing a slight 
improvement over both Logistic Regression and SVM in terms of overall performance.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Despite this improvement, the F1-score remains relatively low, indicating that the model still struggles to maintain a 
balanced trade-off between precision and recall. This suggests that the separation between <b>Toluene</b> and 
<b>2-butanone</b> remains challenging, even when using a more flexible, non-linear model.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
A notable observation is the relatively high <b>standard deviation</b> across all metrics, which reflects variability 
between folds. This behavior is expected given the limited number of runs and highlights the sensitivity of the model 
to differences between experimental conditions.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, while Random Forest provides a modest performance improvement, the results indicate that increasing model 
complexity alone is not sufficient to fully resolve the classification difficulty. This reinforces the idea that the 
Stage 2 task is inherently more complex and may require more discriminative features or alternative modeling strategies.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
4.4 Gradient Boosting
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Gradient Boosting is evaluated as another ensemble-based approach that builds decision trees sequentially, with each 
new tree attempting to correct the errors made by the previous ones. Unlike Random Forest, which constructs trees 
independently, Gradient Boosting refines the model iteratively and can capture more subtle non-linear relationships.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This approach is particularly relevant for the current task, where distinguishing between <b>Toluene</b> and 
<b>2-butanone</b> may depend on complex interactions among temporal, statistical, and shape-based signal features that 
are not easily captured by simpler models.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
By progressively focusing on harder-to-classify samples, Gradient Boosting has the potential to improve classification 
performance compared to both linear models and independently constructed tree ensembles.
</p>

In [8]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_validate
import pandas as pd

# Gradient Boosting model
gb_model = GradientBoostingClassifier(random_state=42)

# Cross-validation with multiple metrics
gb_results = cross_validate(
    gb_model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False,
    n_jobs=-1
)

# Summary table
gb_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        gb_results["test_accuracy"].mean(),
        gb_results["test_precision"].mean(),
        gb_results["test_recall"].mean(),
        gb_results["test_f1"].mean()
    ],
    "Std": [
        gb_results["test_accuracy"].std(),
        gb_results["test_precision"].std(),
        gb_results["test_recall"].std(),
        gb_results["test_f1"].std()
    ]
})

display(gb_summary)

,Metric,Mean,Std
0,Accuracy,0.724848,0.182160
1,Precision,0.588483,0.384946
2,Recall,0.551772,0.411548
3,F1-score,0.561800,0.406671


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The Gradient Boosting model achieves a <b>mean accuracy of 72.5%</b> and an <b>F1-score of 56.2%</b>, which is slightly 
lower than both Logistic Regression and Random Forest.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Despite its ability to model complex non-linear relationships, the performance does not improve in this case, suggesting 
that the underlying feature space does not provide sufficiently strong or consistent patterns for Gradient Boosting to 
exploit effectively.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Similar to previous models, there is a noticeable gap between accuracy and F1-score, indicating that the model struggles 
to maintain a balanced trade-off between precision and recall. This further confirms the difficulty of distinguishing 
between <b>Toluene</b> and <b>2-butanone</b> based on the available features.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, the relatively high variability across folds highlights sensitivity to differences between runs, reinforcing 
the importance of the group-aware validation strategy used in this study.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results indicate that even more advanced ensemble methods do not significantly improve performance in Stage 2, 
suggesting that the primary limitation lies in the discriminative power of the engineered features rather than the choice 
of model.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
4.5 Model Comparison
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To systematically compare the evaluated models, the cross-validation results are summarized using both 
<b>accuracy</b> and <b>F1-score</b>, together with their corresponding standard deviations across folds.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This comparison provides a comprehensive view of both predictive performance and model stability. Given the limited 
number of runs and the presence of variability between folds, evaluating both mean performance and dispersion is 
essential for selecting a reliable model.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
All models are evaluated under the same group-aware cross-validation framework, ensuring a fair comparison and a 
realistic estimate of generalization performance on unseen runs.
</p>

In [9]:
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Mean Accuracy": [
        lr_results["test_accuracy"].mean(),
        svm_results["test_accuracy"].mean(),
        rf_results["test_accuracy"].mean(),
        gb_results["test_accuracy"].mean()
    ],
    "Mean F1-score": [
        lr_results["test_f1"].mean(),
        svm_results["test_f1"].mean(),
        rf_results["test_f1"].mean(),
        gb_results["test_f1"].mean()
    ],
    "Std Accuracy": [
        lr_results["test_accuracy"].std(),
        svm_results["test_accuracy"].std(),
        rf_results["test_accuracy"].std(),
        gb_results["test_accuracy"].std()
    ]
})

comparison_df = comparison_df.sort_values(
    by="Mean F1-score",
    ascending=False
)

display(comparison_df)

,Model,Mean Accuracy,Mean F1-score,Std Accuracy
2,Random Forest,0.762813,0.594152,0.174510
0,Logistic Regression,0.742159,0.581953,0.180948
1,SVM,0.733890,0.567070,0.187002
3,Gradient Boosting,0.724848,0.561800,0.182160


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Among the evaluated models, <b>Random Forest</b> achieves the highest performance, with a mean accuracy of approximately 
<b>76.3%</b> and an F1-score of <b>59.4%</b>. However, the performance gap between models is relatively small, indicating 
that no single approach provides a clearly dominant solution for this task.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Linear and non-linear models, including Logistic Regression, SVM, and Gradient Boosting, all produce comparable results, 
suggesting that increasing model complexity does not lead to substantial improvements in classification performance.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Across all models, the relatively low F1-scores compared to accuracy indicate difficulty in achieving a balanced 
classification between <b>Toluene</b> and <b>2-butanone</b>. This suggests that the two classes exhibit overlapping 
characteristics in the current feature space.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, the consistently high standard deviation across models reflects variability between folds, highlighting 
the impact of differences between experimental runs and reinforcing the importance of group-aware validation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results indicate that the Stage 2 classification task is inherently more challenging than Stage 1. The 
limiting factor appears to be the discriminative power of the engineered features rather than the choice of model, 
suggesting that further improvements may require more specialized features or alternative representations.
</p>

<h2 style="color:#2E86C1; font-weight:bold;">
5. Hyperparameter Tuning
</h2>

<h3 style="color:#2874A6; font-weight:bold;">
5.1 Logistic Regression Tuning
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
After establishing baseline model performance, the next step is to examine whether <b>Logistic Regression</b> can be improved through targeted hyperparameter tuning. Although the baseline results were modest, Logistic Regression remains an important candidate because it is simple, interpretable, and naturally provides probability estimates.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In this stage, tuning focuses on key parameters that influence regularization strength and optimization behavior. In particular, the regularization parameter <b>C</b> is varied to test different balances between underfitting and overfitting, while the model remains embedded in a scaling pipeline to preserve a leakage-safe evaluation setup.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
All candidate settings are evaluated under the same <b>group-aware cross-validation</b> strategy used in the baseline experiments. This ensures that any observed improvement reflects genuine gains in generalization to unseen runs rather than differences in the validation procedure.
</p>

In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Pipeline
lr_tuned_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Parameter grid
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 50],
    "model__class_weight": [None, "balanced"]
}

# Grid search with group-aware CV
lr_grid = GridSearchCV(
    estimator=lr_tuned_pipeline,
    param_grid=param_grid,
    scoring={
        "f1": "f1",
        "accuracy": "accuracy"
    },
    cv=cv,
    n_jobs=-1,
    refit="f1"   # انتخاب بهترین مدل بر اساس F1
)

# Fit
lr_grid.fit(X, y, groups=groups)

# Best results
print("Best parameters:", lr_grid.best_params_)
print("Best mean CV F1-score:", lr_grid.best_score_)
print("Best mean CV Accuracy:",
      lr_grid.cv_results_["mean_test_accuracy"][lr_grid.best_index_])



Best parameters: {'model__C': 0.01, 'model__class_weight': None}
Best mean CV F1-score: 0.6181778946129399
Best mean CV Accuracy: 0.7898814313002219


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
After hyperparameter tuning, Logistic Regression shows a modest improvement compared to the baseline model. The best configuration is obtained with a relatively small regularization parameter (<b>C = 0.01</b>), indicating that stronger regularization helps improve generalization performance in this dataset.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The tuned model achieves a mean <b>F1-score of approximately 0.62</b> and a mean <b>accuracy of 0.79</b>. This represents a noticeable improvement over the baseline results, suggesting that careful control of model complexity can partially mitigate overfitting and improve robustness across folds.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Interestingly, the best configuration does not require class weighting, indicating that the dataset is sufficiently balanced and that reweighting does not contribute significantly to performance improvement in this case.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Despite these improvements, the overall performance remains limited, confirming that Logistic Regression struggles to fully capture the underlying patterns needed to reliably distinguish between <b>Toluene</b> and <b>2-butanone</b>. This suggests that the limitation is not purely due to model configuration, but also related to the intrinsic difficulty of the feature space.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
5.2 Random Forest Tuning
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Following the Logistic Regression tuning step, Random Forest is further optimized to evaluate whether its performance 
can be improved through hyperparameter tuning. As an ensemble-based model capable of capturing non-linear relationships, 
Random Forest previously demonstrated the strongest baseline performance among the evaluated models.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The tuning process focuses on key parameters that control model complexity and generalization, including the number of 
trees, tree depth, and minimum samples required for splitting. Adjusting these parameters allows the model to balance 
between underfitting and overfitting, particularly important given the limited number of runs and the variability across folds.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
All configurations are evaluated using the same group-aware cross-validation strategy to ensure a fair comparison and 
to maintain a leakage-safe evaluation framework. The goal is to determine whether Random Forest can further improve its 
performance and establish itself as the most suitable model for Stage 2 classification.
</p>

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import pandas as pd

# Pipeline (no scaler needed for RF)
rf_pipeline = Pipeline([
    ("model", RandomForestClassifier(random_state=42))
])

# Parameter grid
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

# Grid search
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring={
        "f1": "f1",
        "accuracy": "accuracy"
    },
    cv=cv,
    n_jobs=-1,
    refit="f1"
)

# Fit
rf_grid.fit(X, y, groups=groups)

# Results
print("Best parameters:", rf_grid.best_params_)
print("Best mean CV F1-score:", rf_grid.best_score_)
print("Best mean CV Accuracy:",
      rf_grid.cv_results_["mean_test_accuracy"][rf_grid.best_index_])


Best parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Best mean CV F1-score: 0.6106014271151886
Best mean CV Accuracy: 0.7781765910482026


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
After hyperparameter tuning, the performance of the Random Forest model does not show a significant improvement compared to its baseline results. The best configuration is obtained with a relatively shallow tree structure (<b>max_depth = 5</b>) and a slightly constrained leaf size (<b>min_samples_leaf = 2</b>), indicating that limiting model complexity helps improve generalization.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The tuned model achieves a mean <b>F1-score of approximately 0.61</b> and a mean <b>accuracy of 0.78</b>, which is slightly lower than the baseline Random Forest performance. This suggests that increasing model flexibility does not lead to better generalization in this dataset.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The fact that a simpler configuration outperforms more complex ones indicates that the model is prone to overfitting when allowed to grow deeper trees. This behavior is consistent with the limited number of runs and the variability across folds.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results confirm that hyperparameter tuning does not substantially improve Random Forest performance, and that the primary limitation lies in the separability of the feature space rather than the model capacity.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
5.3 Gradient Boosting Tuning
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In the final tuning step, Gradient Boosting is optimized to evaluate whether its sequential learning mechanism can 
improve performance through more refined hyperparameter selection. Unlike Random Forest, Gradient Boosting builds trees 
iteratively, allowing each new tree to focus on correcting the errors of the previous ones.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The tuning process focuses on parameters that directly influence model learning dynamics, including the number of trees, 
learning rate, and tree depth. These parameters control how aggressively the model learns from data and how well it can 
capture complex patterns without overfitting.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
As with previous models, all configurations are evaluated under the same group-aware cross-validation framework. This 
ensures that performance improvements, if any, reflect genuine generalization to unseen runs rather than differences in 
evaluation strategy.
</p>

In [17]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import pandas as pd

# Pipeline
gb_pipeline = Pipeline([
    ("model", GradientBoostingClassifier(random_state=42))
])

# Parameter grid
gb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5]
}

# Grid search
gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring={
        "f1": "f1",
        "accuracy": "accuracy"
    },
    cv=cv,
    n_jobs=-1,
    refit="f1"
)

# Fit
gb_grid.fit(X, y, groups=groups)

# Results
print("Best parameters:", gb_grid.best_params_)
print("Best mean CV F1-score:", gb_grid.best_score_)
print("Best mean CV Accuracy:",
      gb_grid.cv_results_["mean_test_accuracy"][gb_grid.best_index_])



Best parameters: {'model__learning_rate': 0.01, 'model__max_depth': 5, 'model__n_estimators': 200}
Best mean CV F1-score: 0.6075766619033974
Best mean CV Accuracy: 0.7570879785366906


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
After hyperparameter tuning, Gradient Boosting shows a moderate improvement compared to its baseline performance. 
The best configuration is achieved with a relatively low learning rate (<b>learning_rate = 0.01</b>), a deeper tree 
structure (<b>max_depth = 5</b>), and a larger number of estimators (<b>n_estimators = 200</b>), indicating that a 
more gradual and controlled learning process benefits this model.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The tuned model achieves a mean <b>F1-score of approximately 0.61</b> and an <b>accuracy of 0.76</b>. While this represents 
an improvement over the baseline Gradient Boosting model, the overall performance remains comparable to the tuned 
Logistic Regression and Random Forest models.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The selected configuration suggests that the model benefits from slower learning and increased model capacity, allowing 
it to capture more subtle patterns in the data. However, the absence of a significant performance gain compared to other 
models indicates that these additional complexities do not substantially improve generalization.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results confirm that even with careful tuning, Gradient Boosting does not provide a clear advantage over 
simpler models. This reinforces the conclusion that the primary limitation of the Stage 2 task lies in the 
discriminative power of the engineered features rather than in model selection or hyperparameter optimization.
</p>

<h2 style="color:#2E86C1; font-weight:bold;">
6. Final Model Selection
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
After comparing the tuned versions of all candidate models, the final model for Stage 2 is selected based on a combination of 
predictive performance, stability, and suitability for probability-based interpretation. Since the objective of this stage is not 
only to assign a class label but also to provide reliable confidence estimates, model selection must consider both classification 
quality and interpretability.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Although the performance differences between tuned models are relatively small, <b>Logistic Regression</b> achieves the highest 
overall <b>F1-score</b> and remains one of the most stable and interpretable approaches. In addition, its probability outputs are 
naturally well suited for the probability-based objective of Stage 2.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
These results indicate that, under the current feature representation, increasing model complexity does not provide a meaningful 
advantage. Instead, a simpler and more strongly regularized linear model offers the best balance between generalization, robustness, 
and downstream usability.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Among the tuned models, <b>Logistic Regression</b> achieves the strongest overall performance, with the highest mean 
<b>F1-score</b> and competitive accuracy. This indicates that it offers the most effective balance between precision and recall 
for distinguishing between <b>Toluene</b> and <b>2-butanone</b> in the current dataset.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
An important finding is that more complex models such as <b>Random Forest</b> and <b>Gradient Boosting</b> do not produce 
meaningful gains over the tuned Logistic Regression model. This suggests that the main limitation of Stage 2 lies not in model 
capacity, but in the degree of overlap between the two gas classes under the existing feature representation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Logistic Regression is also particularly suitable for this stage because it directly provides class probability estimates, 
which align with the confidence-oriented objective of Stage 2. Compared with more complex models, it offers a clearer and 
more interpretable probability-based decision mechanism.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the tuned <b>Logistic Regression</b> model is selected as the final Stage 2 classifier. It provides the best combination 
of predictive performance, stability, interpretability, and probability estimation capability, making it the most appropriate 
choice for downstream deployment within the hierarchical gas classification pipeline.
</p>